# Reto AgroCebada FIRA 2026
## Libreta 02 - Ingenieria Fenologica y Espacial

> *"Traducimos pixeles a agronomia: estres hidrico, senescencia y acumulacion termica."*

### Seccion del Reporte que alimenta esta libreta
| Seccion del Reporte | Contenido generado aqui |
|---|---|
| **Metodologia** | Construccion de bloques tematicos, variables fenologicas, integracion espacial |

### Objetivos de esta libreta
1. Extraer variables climaticas (CHIRPS/CHIRTS-ERA5) y topograficas (INEGI CEM 4.0) desde rasteres GeoTIFF.
2. Construir variables fenologicas interpretables: AUC, Delta maximo, P90, IQR y Mes del maximo.
3. Integrar los bloques tematicos: Suelo/Termico, Vegetacion, Agua, Clima, Topografia, Fisica, Fenologia y Area.
4. Exportar la matriz integrada de caracteristicas lista para la Libreta 03.

### Alcance temporal
Ciclo agricola **abril-octubre de 2025**. Analisis retrospectivo del ciclo completo.

### Dependencia
Esta libreta lee los artefactos generados por la **Libreta 01** y los datos crudos del `DataSet.zip`.

In [5]:
# ============================================================
# 1. CONFIGURACION MAESTRA Y ENTORNO (LIBRETA 02)
# ============================================================
!pip install -q geopandas shapely pyproj pyshp rasterio rasterstats scipy

import os
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'

EN_COLAB = True  # Cambiar a False si el evaluador lo corre en su PC local (GitHub)

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Reto_AgroCebada_2026')
else:
    BASE = Path(os.getcwd())

CARPETAS = {
    'raw':           BASE / 'data' / 'raw',
    '01_calidad':    BASE / 'data' / '01_calidad',
    '02_fenologia':  BASE / 'data' / '02_fenologia',
    '03_validacion': BASE / 'data' / '03_validacion',
    '04_shap':       BASE / 'data' / '04_shap',
    '05_final':      BASE / 'data' / '05_final',
    'models':        BASE / 'models',
    'artifacts':     BASE / 'artifacts',
    'reports':       BASE / 'reports',
}

for carpeta in CARPETAS.values():
    carpeta.mkdir(parents=True, exist_ok=True)

# Funciones auxiliares
def guardar_csv(df, nombre, carpeta='raw'):
    ruta = CARPETAS[carpeta] / nombre
    df.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f"[OK] Guardado: {ruta}")
    return ruta

def cargar_csv(nombre, carpeta='raw', **kwargs):
    ruta = CARPETAS[carpeta] / nombre
    assert ruta.exists(), f"[ERROR] No existe: {ruta}"
    df = pd.read_csv(ruta, **kwargs)
    print(f"[OK] Cargado: {nombre} ({df.shape[0]} filas x {df.shape[1]} cols)")
    return df

def guardar_figura(fig, nombre, carpeta='reports'):
    ruta = CARPETAS[carpeta] / nombre
    fig.savefig(ruta, dpi=200, bbox_inches='tight')
    print(f"[OK] Grafico guardado: {ruta}")
    return ruta

def listar_archivos(carpeta='raw'):
    ruta = CARPETAS[carpeta]
    archivos = sorted(ruta.iterdir()) if ruta.exists() else []
    if not archivos:
        print(f"Carpeta '{carpeta}' vacia")
    else:
        print(f"Carpeta '{carpeta}': {len(archivos)} archivos")
        for a in archivos:
            tamano = a.stat().st_size
            unidad = 'MB' if tamano > 1e6 else 'KB' if tamano > 1e3 else 'B'
            valor = tamano / 1e6 if tamano > 1e6 else tamano / 1e3 if tamano > 1e3 else tamano
            print(f"   - {a.name} ({valor:.1f} {unidad})")
    return archivos

# Parametros del proyecto
INICIO = '2025-04-01'
FIN = '2025-10-31'
MESES = [f'2025-{m:02d}' for m in range(4, 11)]
NOMBRES_MESES = ['Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre']

print("[OK] Configuracion completada")
print(f"     Ventana temporal: {INICIO} a {FIN}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Configuracion completada
     Ventana temporal: 2025-04-01 a 2025-10-31


In [6]:
# ============================================================
# 2. CARGA DE DATOS BASE
# ============================================================
RAW = CARPETAS['raw'] / 'extraido' / 'DATASET_RETO_AGRO_2026'

# Tabla Maestra
ruta_maestro = RAW / 'ID_area_rendimiento_70_30_Reto_AgroCebada.csv'
df_maestro = pd.read_csv(ruta_maestro)
df_maestro['ID_POLIGONO'] = df_maestro['ID_POLIGONO'].astype(str).str.strip()
columnas_vacias = [c for c in df_maestro.columns if c.startswith('Unnamed:') and df_maestro[c].isna().all()]
if columnas_vacias:
    df_maestro = df_maestro.drop(columns=columnas_vacias)

# Dataset PRO (Planet)
df_pro = pd.read_csv(RAW / 'Conjunto_datos_PRO_AgroCebada.csv', dtype={'ID_POLIGONO': str})
df_pro['ID_POLIGONO'] = df_pro['ID_POLIGONO'].str.strip()

# Dataset Basico (Sentinel-2 / Landsat)
df_basico = pd.read_csv(RAW / 'Conjunto_datos_BASICO_AgroCebada2026.csv', dtype={'ID_POLIGONO': str})
df_basico['ID_POLIGONO'] = df_basico['ID_POLIGONO'].str.strip()

# Shapefile - Busqueda recursiva
shp_files = list(set(list((CARPETAS['raw'] / 'extraido').rglob('*.shp')) + list((CARPETAS['raw'] / 'extraido').rglob('*.SHP'))))
assert len(shp_files) >= 1, f"No se encontro archivo .shp en {RAW}"
gdf_parcelas = gpd.read_file(shp_files[0])

# Normalizar columna ID
columnas_id = [c for c in gdf_parcelas.columns if 'ID_POL' in c.upper()]
if columnas_id:
    gdf_parcelas = gdf_parcelas.rename(columns={columnas_id[0]: 'ID_POLIGONO'})
gdf_parcelas['ID_POLIGONO'] = gdf_parcelas['ID_POLIGONO'].astype(str).str.strip()

# Separacion entrenamiento/prediccion
df_entrenamiento = df_maestro[df_maestro['CONJUNTO'] == 'ENTRENAMIENTO'].copy()
df_prediccion = df_maestro[df_maestro['CONJUNTO'] == 'PREDICCION'].copy()

print(f"[OK] Maestro:       {len(df_maestro)} parcelas")
print(f"[OK] PRO:           {len(df_pro)} capturas")
print(f"[OK] Basico:        {len(df_basico)} capturas")
print(f"[OK] Shapefile:     {len(gdf_parcelas)} geometrias")
print(f"[OK] Entrenamiento: {len(df_entrenamiento)} | Prediccion: {len(df_prediccion)}")

[OK] Maestro:       197 parcelas
[OK] PRO:           47804 capturas
[OK] Basico:        107666 capturas
[OK] Shapefile:     197 geometrias
[OK] Entrenamiento: 138 | Prediccion: 59


### Extraccion de Variables Climaticas y Topograficas

Las fuentes raster del reto incluyen:
- **CHIRPS**: precipitacion mensual (mm/mes), ~5 km (0.05 grados).
- **CHIRTS-ERA5**: temperatura minima y maxima mensual (grados C), ~5 km (0.05 grados).
- **INEGI CEM 4.0**: elevacion (m s.n.m.) y pendiente (grados), 120 m.

**Metodo de extraccion implementado:** `rasterstats.zonal_stats(..., stats="mean", all_touched=True)` tras reproyectar las parcelas al CRS de cada raster. Calcula el promedio simple de los pixeles que tocan el poligono. Con pixeles climaticos de ~5 km, la mayoria de las parcelas caen en uno o pocos pixeles, de modo que parcelas vecinas reciben valores similares.

**Manejo de NoData:** Se utiliza el valor declarado en cada archivo (o -9999 si no lo especifica) y, como salvaguarda, cualquier media menor o igual a -9000 se convierte en `NaN`.

In [ ]:
# ============================================================
# 3. EXTRACCION RASTER (CLIMA + TOPOGRAFIA)
# ============================================================
print("1. Localizando los ZIPs internos de clima y topografia...")
RAW_INTERNO = RAW if RAW.exists() else CARPETAS['raw'] / 'extraido'

carpeta_raster = CARPETAS['raw'] / 'raster_extraidos'
carpeta_raster.mkdir(parents=True, exist_ok=True)

zips_internos = list(RAW_INTERNO.glob('*.zip'))
zips_clima_topo = [z for z in zips_internos
                   if any(k in str(z).lower() for k in ['chirps', 'topografia', 'temperatura', 'inegi', 'cem'])]

if not zips_clima_topo:
    print("[AVISO] No se encontraron los ZIPs de clima/topografia. ZIPs disponibles:")
    for z in zips_internos:
        print(f"   - {z.name}")

for archivo in zips_clima_topo:
    print(f"   - Descomprimiendo: {archivo.name}")
    with zipfile.ZipFile(archivo, 'r') as zip_ref:
        zip_ref.extractall(carpeta_raster)

print("\n2. Buscando archivos .tif...")
archivos_tif = sorted(carpeta_raster.glob('**/*.tif'))
print(f"   Se encontraron {len(archivos_tif)} GeoTIFF (esperados: 146 = 48 PREC + 48 Tmin + 48 Tmax + 2 topografia).")
assert len(archivos_tif) > 0, "No hay archivos .tif; revisa la estructura de los ZIP."

if gdf_parcelas.crs is None:
    gdf_parcelas = gdf_parcelas.set_crs('EPSG:4326')

print("\n3. Extrayendo estadisticas zonales (promedio simple, all_touched=True)...")
columnas_raster = {}
cache_crs = {}
errores = []

for tif in archivos_tif:
    nombre_columna = tif.stem.replace(' ', '_') + '_mean'
    try:
        with rasterio.open(tif) as src:
            raster_crs = src.crs
            nodata_archivo = src.nodata
        if raster_crs is None:
            errores.append((tif.name, 'sin CRS'))
            continue
        clave = raster_crs.to_string()
        if clave not in cache_crs:
            cache_crs[clave] = gdf_parcelas.to_crs(raster_crs)
        nodata = nodata_archivo if nodata_archivo is not None else -9999
        stats = zonal_stats(cache_crs[clave], str(tif), stats='mean', all_touched=True, nodata=nodata)
        valores = np.array([s['mean'] if (s and s.get('mean') is not None) else np.nan for s in stats], dtype=float)
        valores[valores <= -9000] = np.nan
        columnas_raster[nombre_columna] = valores
    except Exception as e:
        errores.append((tif.name, str(e)))

ids_parcelas = gdf_parcelas['ID_POLIGONO'].astype(str).str.strip().reset_index(drop=True)
df_clima_topo = pd.concat([ids_parcelas.to_frame('ID_POLIGONO'), pd.DataFrame(columnas_raster)], axis=1)

print(f"\n[OK] Extraccion completada: {df_clima_topo.shape[1] - 1} variables, {len(df_clima_topo)} parcelas.")
if errores:
    print(f"[AVISO] {len(errores)} archivos con error: {errores[:5]}")

pct_nan = df_clima_topo.drop(columns='ID_POLIGONO').isna().mean().sort_values(ascending=False)
print(f"   Columnas con NaN: {(pct_nan > 0).sum()} (maximo {pct_nan.iloc[0] * 100:.1f}% en {pct_nan.index[0]})")

guardar_csv(df_clima_topo, 'datos_clima_topografia.csv', 'raw')

1. Localizando los ZIPs internos de clima y topografia...
   - Descomprimiendo: Reto_AgroCebada_CHIRPS_Precipitacion_2022_2025.zip
   - Descomprimiendo: Reto_AgroCebada_Temperatura_2022_2025_RECORTADA_CHIRPS.zip
   - Descomprimiendo: Reto_AgroCebada_Topografia_INEGI_CEM4.zip

2. Buscando archivos .tif...
   Se encontraron 146 GeoTIFF (esperados: 146 = 48 PREC + 48 Tmin + 48 Tmax + 2 topografia).

3. Extrayendo estadisticas zonales (promedio simple, all_touched=True)...


In [ ]:
# ============================================================
# 4. FILTRAR CLIMA A LA VENTANA DEL CICLO AGRICOLA (ABR-OCT 2025)
# ============================================================
df_clima = df_clima_topo.copy()
df_clima['ID_POLIGONO'] = df_clima['ID_POLIGONO'].astype(str).str.strip()

cols_clima = [c for c in df_clima.columns if c != 'ID_POLIGONO']
print(f"Variables climaticas/topograficas totales: {len(cols_clima)}")

meses_cultivo = [f'2025_{m:02d}' for m in range(4, 11)]
cols_validas = [c for c in cols_clima
                if any(m in c for m in meses_cultivo) or 'Elevacion' in c or 'Pendiente' in c]

n_prec = sum(c.startswith('PREC_') for c in cols_validas)
n_tmin = sum(c.startswith('Tmin_') for c in cols_validas)
n_tmax = sum(c.startswith('Tmax_') for c in cols_validas)
n_topo = sum(('Elevacion' in c) or ('Pendiente' in c) for c in cols_validas)
print(f"Variables en la ventana del ciclo: {len(cols_validas)} "
      f"(PREC={n_prec}, Tmin={n_tmin}, Tmax={n_tmax}, topografia={n_topo})")

assert (n_prec, n_tmin, n_tmax) == (7, 7, 7), (
    f"Se esperaban 7 meses de PREC, Tmin y Tmax (abril-octubre 2025); hay {(n_prec, n_tmin, n_tmax)}.")
assert n_topo == 2, f"Se esperaban 2 variables topograficas (elevacion y pendiente); hay {n_topo}."

df_clima_filtrado = df_clima[['ID_POLIGONO'] + sorted(cols_validas)].copy().set_index('ID_POLIGONO')

print("\nColumnas seleccionadas:")
for c in df_clima_filtrado.columns:
    print(f"   - {c}")

In [ ]:
# ============================================================
# 5. VARIABLES CLIMATICAS DERIVADAS ("FISICAS")
# ============================================================
def _cols(prefijo, meses):
    return [f'{prefijo}_2025_{m:02d}_mean' for m in meses]

MESES_CICLO = range(4, 11)       # abril-octubre
MESES_LLENADO = range(7, 10)     # julio-septiembre (llenado de grano)

d = df_clima_filtrado
prec_ciclo = d[_cols('PREC', MESES_CICLO)].sum(axis=1, skipna=False)
prec_llenado = d[_cols('PREC', MESES_LLENADO)].sum(axis=1, skipna=False)
tmax_ciclo = d[_cols('Tmax', MESES_CICLO)].mean(axis=1)
tmin_ciclo = d[_cols('Tmin', MESES_CICLO)].mean(axis=1)
tmax_llenado = d[_cols('Tmax', MESES_LLENADO)].mean(axis=1)

df_fisica_clima = pd.DataFrame({
    'fisica__prec_acum_abr_oct': prec_ciclo,
    'fisica__prec_acum_jul_sep': prec_llenado,
    'fisica__tmax_media_abr_oct': tmax_ciclo,
    'fisica__tmin_media_abr_oct': tmin_ciclo,
    'fisica__amplitud_termica_abr_oct': tmax_ciclo - tmin_ciclo,
    'fisica__cociente_estres_jul_sep': prec_llenado / tmax_llenado.where(tmax_llenado > 0),
}, index=d.index)

print(f"[OK] Variables fisicas de clima: {df_fisica_clima.shape[1]}")
print(df_fisica_clima.describe().T[['count', 'mean', 'min', 'max']].round(2).to_string())

### Construccion de Variables Fenologicas

Para cada indice satelital, extraemos metricas que capturan la dinamica del cultivo a lo largo del ciclo:

| Metrica | Interpretacion agronomica (aproximada) |
|---|---|
| **AUC** | Suma de los valores mensuales (mediana mensual del promedio espacial por captura). Proxy de acumulacion del dosel. |
| **Delta maximo** | Mayor cambio absoluto entre meses consecutivos (velocidad de crecimiento/senescencia). |
| **P90** | Percentil 90 de la serie mensual (pico de vigor). |
| **IQR** | Rango intercuartilico de la serie mensual (variabilidad intra-ciclo). |
| **Mes del maximo** | Numero de mes (4-10) en que ocurre el maximo; fenologia aproximada. |

**Notas metodologicas:**
- Se calculan sobre series **mensuales** construidas mediante la mediana de las capturas de cada mes.
- Se exige un minimo de 3 meses con dato por parcela; las parcelas con menos meses quedan con `NaN` en esa variable.

In [ ]:
# ============================================================
# 6. CONSTRUCCION DE VARIABLES FENOLOGICAS
# ============================================================
df_basico['fecha_captura'] = pd.to_datetime(df_basico['fecha_captura'], format='%d/%m/%Y')
df_pro['fecha_captura'] = pd.to_datetime(df_pro['fecha_captura'], format='%d/%m/%Y')

ventana_inicio = pd.Timestamp(INICIO)
ventana_fin = pd.Timestamp(FIN)

df_basico_ventana = df_basico[
    (df_basico['fecha_captura'] >= ventana_inicio) &
    (df_basico['fecha_captura'] <= ventana_fin)
].copy()
df_basico_ventana['mes'] = df_basico_ventana['fecha_captura'].dt.to_period('M').astype(str)

df_pro_ventana = df_pro[
    (df_pro['fecha_captura'] >= ventana_inicio) &
    (df_pro['fecha_captura'] <= ventana_fin)
].copy()
df_pro_ventana['mes'] = df_pro_ventana['fecha_captura'].dt.to_period('M').astype(str)

sufijos = ['_promedio', '_std', '_min', '_max']
columnas_meta = ['ID_POLIGONO', 'fecha_captura', 'sensor', 'mes', 'porcentaje_nubosidad']

def extraer_indices(df):
    cols = [c for c in df.columns if c not in columnas_meta]
    indices = set()
    for col in cols:
        for suf in sufijos:
            if suf in col:
                indices.add(col.replace(suf, ''))
                break
    return sorted(indices)

indices_basico = extraer_indices(df_basico_ventana)
indices_pro = extraer_indices(df_pro_ventana)

print(f"Indices Basico: {indices_basico}")
print(f"Indices PRO:    {indices_pro}")

def calcular_fenologia(df, indice, columna_promedio):
    """Calcula AUC, Delta, P90, IQR y Mes del maximo para un indice."""
    resultados = []

    for id_poligono, grupo in df.groupby('ID_POLIGONO'):
        if columna_promedio not in grupo.columns:
            continue

        serie = grupo.groupby('mes')[columna_promedio].median().dropna()
        if len(serie) < 3:
            continue

        valores = serie.values
        meses_serie = serie.index.tolist()

        auc = valores.sum()
        delta_max = np.max(np.abs(np.diff(valores)))
        p90 = np.percentile(valores, 90)
        iqr = np.percentile(valores, 75) - np.percentile(valores, 25)
        mes_maximo = int(meses_serie[int(np.argmax(valores))][-2:])

        resultados.append({
            'ID_POLIGONO': id_poligono,
            f'fenologia__{indice}__auc': auc,
            f'fenologia__{indice}__delta_max': delta_max,
            f'fenologia__{indice}__p90': p90,
            f'fenologia__{indice}__iqr': iqr,
            f'fenologia__{indice}__mes_maximo': mes_maximo,
        })

    return pd.DataFrame(resultados)

dfs_fenologia = []
for indice in indices_basico:
    col_prom = f'{indice}_promedio'
    if col_prom in df_basico_ventana.columns:
        df_fen = calcular_fenologia(df_basico_ventana, indice, col_prom)
        if not df_fen.empty:
            dfs_fenologia.append(df_fen)
            print(f"   [OK] {indice}: {len(df_fen)} parcelas con fenologia")

for indice in indices_pro:
    col_prom = f'{indice}_promedio'
    if col_prom in df_pro_ventana.columns:
        df_fen = calcular_fenologia(df_pro_ventana, f'PRO_{indice}', col_prom)
        if not df_fen.empty:
            dfs_fenologia.append(df_fen)
            print(f"   [OK] PRO_{indice}: {len(df_fen)} parcelas con fenologia")

if dfs_fenologia:
    df_fenologia = dfs_fenologia[0]
    for df in dfs_fenologia[1:]:
        df_fenologia = df_fenologia.merge(df, on='ID_POLIGONO', how='outer')
    df_fenologia = df_fenologia.set_index('ID_POLIGONO')
else:
    df_fenologia = pd.DataFrame()

print(f"\n[OK] Variables fenologicas construidas: {df_fenologia.shape[1]} columnas")
print(f"     Parcelas con fenologia: {len(df_fenologia)}")

### Integracion de los Bloques Tematicos

| Bloque | Fuente | Variables |
|---|---|---|
| **Suelo/Termico** | Sentinel-2 / Landsat (CRC, MCRC, NDSVI, SRNDI, NDTI, STI, MRA) | Mediana temporal del promedio espacial |
| **Vegetacion** | Basico (NDVI, EVI, LAI, FAPAR, SAVI) + PRO (NDVI, EVI, LAI, MSAVI) | Mediana temporal del promedio espacial |
| **Agua** | Sentinel-2 / Landsat (NDWI, DSWI4, DSWI5) | Mediana temporal del promedio espacial |
| **Fenologia** | Todos los indices | AUC, delta max., P90, IQR, mes del maximo |
| **Clima** | CHIRPS + CHIRTS (abril-octubre 2025) | 7 meses x (PREC, Tmin, Tmax) = 21 |
| **Topografia** | INEGI CEM 4.0 | Elevacion y pendiente |
| **Fisica** | Derivadas de fenologia y clima | Integral del dosel (FAPAR, NDVI), mes pico de NDVI, precipitacion acumulada, temperaturas medias, amplitud termica y cociente de estres |
| **Area** | Tabla maestra | `AREA_HA` |

In [ ]:
# ============================================================
# 7. INTEGRACION DE LOS BLOQUES TEMATICOS
# ============================================================
def agregar_mediana(df, indices, plantilla, cols_out, etiqueta):
    """Mediana temporal del promedio espacial por parcela; avisa si el indice no existe."""
    for idx in indices:
        col_prom = f'{idx}_promedio'
        if col_prom not in df.columns:
            print(f"   [AVISO] {etiqueta}: el indice '{idx}' no existe en el dataset; se omite")
            continue
        agg = df.groupby('ID_POLIGONO')[col_prom].median()
        cols_out.append(agg.rename(plantilla.format(idx=idx)))

# Bloque 1: Suelo/Termico (Basico)
cols_suelo = []
agregar_mediana(df_basico_ventana, ['crc', 'mcrc', 'ndsvi', 'srndi', 'ndti', 'sti', 'mra'],
                'suelo_termico__{idx}__mediana', cols_suelo, 'Basico')

# Bloque 2: Vegetacion (Basico + PRO)
cols_veg = []
agregar_mediana(df_basico_ventana, ['ndvi', 'evi', 'lai', 'fapar', 'savi'],
                'vegetacion__basico__{idx}__mediana', cols_veg, 'Basico')
agregar_mediana(df_pro_ventana, ['ndvi', 'evi', 'lai', 'msavi'],
                'vegetacion__pro__{idx}__mediana', cols_veg, 'PRO')

# Bloque 3: Agua (Basico)
cols_agua = []
agregar_mediana(df_basico_ventana, ['ndwi', 'dswi4', 'dswi5'],
                'agua__{idx}__mediana', cols_agua, 'Basico')

# Bloque 4: Entorno (clima + topografia) y variables fisicas de clima
cols_entorno = df_clima_filtrado.copy()

# Consolidar bloques
dfs_bloques = []
for lista in (cols_suelo, cols_veg, cols_agua):
    if lista:
        dfs_bloques.append(pd.concat(lista, axis=1))
dfs_bloques.append(cols_entorno)
dfs_bloques.append(df_fisica_clima)
if not df_fenologia.empty:
    dfs_bloques.append(df_fenologia)

X_integrado = dfs_bloques[0]
for df_b in dfs_bloques[1:]:
    X_integrado = X_integrado.join(df_b, how='outer')

# Variables fisicas derivadas de la fenologia
for nombre, origen in [('fisica__integral_dosel_fapar', 'fenologia__fapar__auc'),
                       ('fisica__integral_dosel_ndvi', 'fenologia__ndvi__auc'),
                       ('fisica__mes_pico_ndvi', 'fenologia__ndvi__mes_maximo')]:
    if origen in X_integrado.columns:
        X_integrado[nombre] = X_integrado[origen]
    else:
        print(f"   [AVISO] No existe {origen}; no se crea {nombre}")

# Area
X_integrado['AREA_HA'] = df_maestro.set_index('ID_POLIGONO')['AREA_HA']

print("=" * 60)
print("MATRIZ INTEGRADA DE CARACTERISTICAS")
print("=" * 60)
print(f"Parcelas:          {len(X_integrado)}")
print(f"Variables totales: {X_integrado.shape[1]}")

n_suelo = len([c for c in X_integrado.columns if c.startswith('suelo_termico')])
n_veg = len([c for c in X_integrado.columns if c.startswith('vegetacion')])
n_agua = len([c for c in X_integrado.columns if c.startswith('agua')])
n_clima = len([c for c in X_integrado.columns if c.startswith(('PREC_', 'Tmin_', 'Tmax_'))])
n_topografia = len([c for c in X_integrado.columns if c.startswith(('Elevacion', 'Pendiente'))])
n_entorno = n_clima + n_topografia
n_fisica = len([c for c in X_integrado.columns if c.startswith('fisica__')])
n_fenologia = len([c for c in X_integrado.columns if c.startswith('fenologia')])

print("\nBloques:")
print(f"   Suelo/Termico: {n_suelo} variables")
print(f"   Vegetacion:    {n_veg} variables")
print(f"   Agua:          {n_agua} variables")
print(f"   Clima:         {n_clima} variables (esperadas: 21)")
print(f"   Topografia:    {n_topografia} variables (esperadas: 2)")
print(f"   Fisica:        {n_fisica} variables")
print(f"   Fenologia:     {n_fenologia} variables")
print(f"   Area:          1 variable")

assert n_clima == 21, f"El bloque de clima debe tener 21 variables y tiene {n_clima}."
assert n_topografia == 2, f"La topografia debe tener 2 variables y tiene {n_topografia}."

In [ ]:
# ============================================================
# 8. DIAGNOSTICO DE CALIDAD DE LA MATRIZ INTEGRADA
# ============================================================
print("=" * 60)
print("DIAGNOSTICO DE CALIDAD DE LA MATRIZ INTEGRADA")
print("=" * 60)

print("\nProporcion de valores faltantes por bloque:")
for prefijo, nombre in [('suelo_termico', 'Suelo/Termico'),
                         ('vegetacion', 'Vegetacion'),
                         ('agua', 'Agua'),
                         ('fenologia', 'Fenologia')]:
    cols_bloque = [c for c in X_integrado.columns if c.startswith(prefijo)]
    if cols_bloque:
        pct_faltante = X_integrado[cols_bloque].isna().mean().mean() * 100
        print(f"   {nombre}: {pct_faltante:.1f}% faltante")

cols_entorno_check = [c for c in X_integrado.columns if any(k in c for k in ['PREC', 'Tmax', 'Tmin', 'Elevacion', 'Pendiente'])]
if cols_entorno_check:
    pct_entorno = X_integrado[cols_entorno_check].isna().mean().mean() * 100
    print(f"   Entorno: {pct_entorno:.1f}% faltante")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribucion de AUC de NDVI
ax = axes[0, 0]
col_auc_ndvi = [c for c in X_integrado.columns if 'ndvi' in c and 'auc' in c]
if col_auc_ndvi:
    X_integrado[col_auc_ndvi[0]].hist(bins=30, ax=ax, edgecolor='white', color='seagreen')
    ax.set_title(f'Distribucion de {col_auc_ndvi[0]}')
    ax.set_xlabel('AUC')
    ax.set_ylabel('Frecuencia')

# 2. Area de parcelas
ax = axes[0, 1]
X_integrado['AREA_HA'].hist(bins=30, ax=ax, edgecolor='white', color='steelblue')
ax.set_title('Distribucion del Area de Parcelas')
ax.set_xlabel('Area (ha)')
ax.set_ylabel('Frecuencia')

# 3. Proporcion de faltantes por variable
ax = axes[1, 0]
faltantes = X_integrado.isna().mean().sort_values(ascending=False).head(20)
faltantes.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 20 Variables con Mayor Proporcion de Faltantes')
ax.set_xlabel('Proporcion de faltantes')

# 4. Correlacion entre bloques
ax = axes[1, 1]
cols_numericas = X_integrado.select_dtypes(include=[np.number]).columns.tolist()
if len(cols_numericas) > 5:
    corr = X_integrado[cols_numericas].corr().iloc[:10, :10]
    sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
                xticklabels=False, yticklabels=True)
    ax.set_title('Correlacion entre Variables (primeras 10)')

plt.tight_layout()
guardar_figura(fig, '03_diagnostico_matriz_integrada.png', 'reports')
plt.show()

In [ ]:
# ============================================================
# 9. EXPORTACION DE ARTEFACTOS DE LA LIBRETA 02
# ============================================================
OUT = CARPETAS['02_fenologia']

# Matriz integrada completa
guardar_csv(X_integrado.reset_index(), 'X_integrado_completo.csv', '02_fenologia')

# Variables fenologicas
if not df_fenologia.empty:
    guardar_csv(df_fenologia.reset_index(), 'variables_fenologicas.csv', '02_fenologia')

# Clima/topografia filtrado
guardar_csv(df_clima_filtrado.reset_index(), 'clima_topografia_filtrado.csv', '02_fenologia')

# Resumen de la libreta
resumen_libreta02 = {
    'version': '2.0-ingenieria-fenologica',
    'ventana_temporal': {'inicio': INICIO, 'fin': FIN},
    'n_parcelas': int(len(X_integrado)),
    'n_variables_totales': int(X_integrado.shape[1]),
    'bloques': {
        'suelo_termico': n_suelo,
        'vegetacion': n_veg,
        'agua': n_agua,
        'entorno': n_entorno,
        'clima': n_clima,
        'topografia': n_topografia,
        'fisica': n_fisica,
        'fenologia': n_fenologia,
        'area': 1
    },
    'metodo_extraccion': 'rasterstats_zonal_mean_all_touched_sin_ponderar_por_area',
    'metodo_indices_satelitales': 'mediana_temporal_del_promedio_espacial_por_captura',
    'alcance': 'retrospectivo; solo ciclo abril-octubre 2025'
}

ruta_resumen = OUT / 'resumen_libreta02.json'
with open(ruta_resumen, 'w', encoding='utf-8') as f:
    json.dump(resumen_libreta02, f, ensure_ascii=False, indent=2)
print(f"[OK] Guardado: {ruta_resumen}")

print("\n" + "=" * 60)
print("ARCHIVOS GENERADOS EN ESTA LIBRETA")
print("=" * 60)
listar_archivos('02_fenologia')

print("\n" + "=" * 60)
print("[OK] LIBRETA 02 COMPLETADA")
print("=" * 60)
print("\nArtefactos clave para el reporte:")
print("   1. X_integrado_completo.csv      -> Matriz de caracteristicas (insumo para Libreta 03)")
print("   2. variables_fenologicas.csv     -> Metricas fenologicas (AUC, Delta, P90, IQR)")
print("   3. clima_topografia_filtrado.csv -> Variables climaticas y topograficas")
print("   4. 03_diagnostico_matriz_integrada.png -> Seccion: Metodologia")
print("   5. resumen_libreta02.json        -> Datos numericos para el texto")
print("\nSiguiente paso: Ejecutar la Libreta 03 - Validacion Robusta y Ablacion")